In [1]:
import os
import time, copy
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import to_hex
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader

from gridnext.gridnet_models import GridNetGraph
from gridnext.training import train_graphwise
from gridnext.utils import all_node_predictions
from gridnext.graph_datasets import compute_adjacency_radius, visium_anndata_to_graphdataset, GraphDataset
from gridnext.plotting import performance_curves, plot_confusion, plot_graph_predictions

/Users/aidandaly/miniforge3/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/aidandaly/miniforge3/lib/python3.10/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
/Users/aidandaly/miniforge3/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [2]:
data_dir = '/Volumes/Aidan_NYGC/Visium/Human_SC_Gridnext'

adata = sc.read_h5ad(os.path.join(data_dir, 'model_predictions', 'scbert', 'adata_human_spc_scbert_logits_p0.h5ad'))

## Compute adjacency, edge weights

In [3]:
# Compute adjacency matrix, assigning all spots of unit distance (within Visium array) as neighbors
A, w = compute_adjacency_radius(adata, spatial_key='spatial', x_col='x', y_col='y', batch_col='array', 
                                pseudo_hex=True, radius=1.01)

In [4]:
# Check that all neighbors are from the same sample/array
for i in range(len(A)):
    assert adata.obs['array'].iloc[A[0,i]] == adata.obs['array'].iloc[A[1,i]]

In [5]:
# Print examples of spots (in Visium pseudo-hex coordinates) and their assigned neighbors
for i in range(15):
    print(adata.obs[['x', 'y']].iloc[i].values)
    for j in A[1][np.where(A[0]==i)]:
        print('\t', adata.obs[['x', 'y']].iloc[j].values)

[73  1]
	 [72  2]
	 [74  2]
[83  1]
	 [82  2]
	 [84  2]
[63  3]
	 [65  3]
	 [64  4]
[65  3]
	 [63  3]
	 [67  3]
	 [64  4]
	 [66  4]
[67  3]
	 [65  3]
	 [69  3]
	 [66  4]
	 [68  4]
[69  3]
	 [67  3]
	 [71  3]
	 [68  4]
	 [70  4]
[71  3]
	 [69  3]
	 [72  2]
	 [73  3]
	 [70  4]
	 [72  4]
[72  2]
	 [73  1]
	 [71  3]
	 [73  3]
	 [74  2]
[73  3]
	 [71  3]
	 [72  2]
	 [74  2]
	 [75  3]
	 [72  4]
	 [74  4]
[74  2]
	 [73  1]
	 [72  2]
	 [73  3]
	 [75  3]
	 [76  2]
[75  3]
	 [73  3]
	 [74  2]
	 [76  2]
	 [77  3]
	 [74  4]
	 [76  4]
[76  2]
	 [74  2]
	 [75  3]
	 [77  3]
	 [78  2]
[77  3]
	 [75  3]
	 [76  2]
	 [78  2]
	 [79  3]
	 [76  4]
	 [78  4]
[78  2]
	 [76  2]
	 [77  3]
	 [79  3]
	 [80  2]
[79  3]
	 [77  3]
	 [78  2]
	 [80  2]
	 [81  3]
	 [78  4]
	 [80  4]


## Convert to PyTorch Graph Data

In [6]:
gdat, batch_names = visium_anndata_to_graphdataset(adata, annot_col='annotation', 
                            spatial_key='spatial', pseudo_hex=True, x_col='x', y_col='y', batch_col='array')

Skipping batch V033-CGND-HRA-02527-A (4 nodes) -- invalid graph


In [7]:
# Train/test splits used in CNN-based model:
val_donors = ['CGND-HRA-00614', 'CGND-HRA-02907', 'CGND-HRC-00023', 'CGND-HRA-02525', 'CGND-HRA-01377']

df = pd.read_csv(os.path.join(data_dir, 'Spinal_Cord_Lumbar_Annotated.csv'), sep=',', index_col=9)
val_arrs = [x.replace('_', '-') for x in df.index if df.loc[x, 'Internal ID'] in val_donors]
train_arrs = [x.replace('_', '-') for x in df.index if df.loc[x, 'Internal ID'] not in val_donors]

gdat_train, gdat_val = [],[]
for b, g in zip(batch_names, gdat):
    if b in val_arrs:
        gdat_val.append(g)
    else:
        gdat_train.append(g)

gdat_train = GraphDataset(gdat_train)
gdat_val = GraphDataset(gdat_val)

print(len(gdat_train), 'train graphs', len(gdat_val), 'validation graphs')

133 train graphs 22 validation graphs


## Training GCN

In [ ]:
f_dim = gdat_train.num_node_features
n_classes = gdat_train.num_classes
num_epochs = 50

f = nn.Identity()
g = GridNetGraph(f, n_classes, use_bn=True)

dataloaders = {'train': DataLoader(gdat_train, batch_size=4, shuffle=True),
               'val': DataLoader(gdat_val, batch_size=4)}
optimizer = torch.optim.Adam(g.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

g, train_history, val_history = train_graphwise(g, dataloaders, loss_fn, optimizer, num_epochs=50)

Epoch 0/49
----------
train Loss: 1.9938 Acc: 0.3843
val Loss: 1.5944 Acc: 0.5567
Epoch 1/49
----------
train Loss: 1.3963 Acc: 0.6485
val Loss: 1.3256 Acc: 0.6114
Epoch 2/49
----------
train Loss: 1.1292 Acc: 0.6943
val Loss: 1.2027 Acc: 0.6247
Epoch 3/49
----------
train Loss: 1.0116 Acc: 0.7202
val Loss: 1.1587 Acc: 0.6487
Epoch 4/49
----------
train Loss: 0.9042 Acc: 0.7395
val Loss: 1.0891 Acc: 0.6579
Epoch 5/49
----------
train Loss: 0.8667 Acc: 0.7527
val Loss: 1.1793 Acc: 0.6193
Epoch 6/49
----------
train Loss: 0.7977 Acc: 0.7654
val Loss: 1.1005 Acc: 0.6484
Epoch 7/49
----------
train Loss: 0.7972 Acc: 0.7602
val Loss: 1.1123 Acc: 0.6478
Epoch 8/49
----------
train Loss: 0.7513 Acc: 0.7695
val Loss: 1.1326 Acc: 0.6252
Epoch 9/49
----------
train Loss: 0.7269 Acc: 0.7755
val Loss: 1.0480 Acc: 0.6629
Epoch 10/49
----------
train Loss: 0.7388 Acc: 0.7724
val Loss: 1.0870 Acc: 0.6429
Epoch 11/49
----------
train Loss: 0.7244 Acc: 0.7716
val Loss: 1.1862 Acc: 0.6168
Epoch 12/49
--

In [ ]:
fig = plt.figure()
plt.title('Training GCN on scBERT logits')
plt.plot(train_history, label='train')
plt.plot(val_history, label='val')
plt.xlabel('Training epochs')
plt.ylabel('Error')
plt.legend()
plt.show()

### Evaluating GCN

In [ ]:
# Softmax & predictions from GCN model
val_true_gcn, val_pred_gcn, val_smax_gcn = all_node_predictions(dataloaders['val'], g)

# Softmax & predictions from scBERT model (prior to applying GCN)
val_true_scbert, val_pred_scbert, val_smax_scbert = all_node_predictions(dataloaders['val'], g, f_only=True)

In [ ]:
class_names = LabelEncoder().fit(adata.obs['annotation']).classes_

fig, ax, macro_auroc, macro_auprc = performance_curves(val_true_gcn, [val_smax_scbert, val_smax_gcn], 
                                                       class_names, ['scBERT', 'GCN(scBERT)'])
plt.suptitle('GCN, scBERT features', fontsize=16)
plt.show()

In [ ]:
fig, ax = plot_confusion(val_true_scbert, val_pred_scbert, class_names)
fig.suptitle('scBERT')
plt.show()

fig, ax = plot_confusion(val_true_gcn, val_pred_gcn, class_names)
fig.suptitle('GCN(scBERT)')
plt.show()

In [ ]:
# Dummy model that applies the "identity" transform to all graph nodes
# -> To visiualize scBERT logits -- used as node features in this example -- directly
class GraphIdentity(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, graph):
        return graph.x

ident = GraphIdentity()
ax0 = plot_graph_predictions(gdat_val[0], ident, class_names)
ax0[1].set_title('scBERT')

ax1 = plot_graph_predictions(gdat_val[0], g, class_names)
ax1[1].set_title('GCN')

In [ ]:
gdat_train[0].edge_attr.squeeze()